In [ ]:
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Load dataset
try:
    df = pd.read_csv('/content/twitter_training.csv', names=['Tweet_ID', 'Entity', 'Sentiment', 'Tweet_Text'])
except FileNotFoundError:
    print("Error: 'twitter_training.csv' not found. Please ensure the file is uploaded to '/content/'.")
    exit()

# Remove missing data
df.dropna(subset=['Tweet_Text', 'Sentiment'], inplace=True)

# Filter for relevant sentiments
df = df[df['Sentiment'].isin(['Positive', 'Negative'])]

# Map 0 and 1
df['Sentiment'] = df['Sentiment'].map({'Negative': 0, 'Positive': 1})

# Prepare data for TF-IDF
x_text = df['Tweet_Text'].astype(str)
y = df['Sentiment']

print(f"Loaded {len(x_text)} Twitter reviews after cleaning and filtering.")

# Training and testing split
x_train_text, x_test_text, y_train, y_test = train_test_split(
    x_text, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF Vectorization
MAX_FEATURES = 5000
vectorizer = TfidfVectorizer(max_features=MAX_FEATURES)

vectorizer.fit(x_train_text)

X_train_tfidf = vectorizer.transform(x_train_text).toarray()
X_test_tfidf = vectorizer.transform(x_test_text).toarray()

# ANN Model
model = Sequential([
    Dense(128, activation="relu", input_shape=(MAX_FEATURES,)),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Train the model
print("\nTraining the model...")
model.fit(X_train_tfidf, y_train, epochs=10, batch_size=32, validation_data=(X_test_tfidf, y_test))

# Evaluate the model
print("\nEvaluating the model...")
loss, accuracy = model.evaluate(X_test_tfidf, y_test)
print(f"Test Accuracy: {accuracy:.4f}")

Loaded 43013 Twitter reviews after cleaning and filtering.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Training the model...
Epoch 1/10
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - accuracy: 0.8508 - loss: 0.3361 - val_accuracy: 0.8969 - val_loss: 0.2421
Epoch 2/10
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 11s 10ms/step - accuracy: 0.9406 - loss: 0.1463 - val_accuracy: 0.9397 - val_loss: 0.1537
Epoch 3/10
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 20s 10ms/step - accuracy: 0.9695 - loss: 0.0689 - val_accuracy: 0.9470 - val_loss: 0.1475
Epoch 4/10
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 11s 10ms/step - accuracy: 0.9772 - loss: 0.0463 - val_accuracy: 0.9505 - val_loss: 0.1458
Epoch 5/10
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 11s 10ms/step - accuracy: 0.9799 - loss: 0.0387 - val_accuracy: 0.9506 - val_loss: 0.1577
Epoch 6/10
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 18s 16ms/step - accuracy: 0.9805 - loss: 0.0354 - val_accuracy: 0.9502 - val_loss: 0.1533
Epoch 7/10
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 15s 11ms/step - accuracy: 0.9808 - loss: 0.0328 - val_accuracy: 0.9482 - val_loss: 0.1739
Epoch 8/10
1076/1076 ━━━━━━━━━━━━━━━━━━━━ 14s 13ms/st

In [ ]:
# User input
print("\nEnter a Sentence:")
user_review = input()

# Transform review using TF-IDF vectorizer
user_review_vector = vectorizer.transform([user_review]).toarray()

# Make prediction
prediction = model.predict(user_review_vector)

if prediction[0][0] > 0.5:
    print(f"Sentiment: Positive (Score: {prediction[0][0]:.4f})")
else:
    print(f"Sentiment: Negative (Score: {prediction[0][0]:.4f})")


Enter a Sentence:
the phone is worst
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Sentiment: Negative (Score: 0.0005)
